# Smart Waste Classification - MobileNetV2 Transfer Learning
Dataset: Kaggle Garbage Classification (Mostafa Mohamed)
Target Classes: Organic Waste, Plastic Waste, Paper Waste
Target Accuracy: >= 85%

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
import numpy as np
import matplotlib.pyplot as plt
import os, shutil
import kagglehub
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow Version:", tf.__version__)

## 1. Download Dataset via Kagglehub & Mapping 3 Kelas

In [ ]:
path = kagglehub.dataset_download("mostafaabla/garbage-classification")
src_base = os.path.join(path, "garbage_classification")

mapping = {
    "biological": "Organic Waste",
    "plastic": "Plastic Waste",
    "paper": "Paper Waste"
}

dataset_dir = "dataset_bilcode"
os.makedirs(dataset_dir, exist_ok=True)

for src_folder, target_name in mapping.items():
    src_dir = os.path.join(src_base, src_folder)
    dst_dir = os.path.join(dataset_dir, target_name)
    os.makedirs(dst_dir, exist_ok=True)
    # Ambil 800 gambar per kelas (total 2400 gambar untuk akurasi tinggi)
    files = os.listdir(src_dir)[:800]
    for img in files:
        shutil.copy(os.path.join(src_dir, img), os.path.join(dst_dir, img))
    print(f"Loaded {len(files)} images for {target_name}")

## 2. Data Loading & Augmentation (80% Train, 20% Test)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)

## 3. Build MobileNetV2 Model

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    layers.Rescaling(1./127.5, offset=-1, input_shape=(224, 224, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 4. Train Model

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

## 5. Evaluation & Metrics (Confusion Matrix & Classification Report)

In [ ]:
y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('confusion_matrix.png')
plt.show()

## 6. Save Model

In [ ]:
model.save("waste_model.keras")
print("Model saved successfully to waste_model.keras")